<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20261020/%E5%9B%9E%E5%B8%B0%E3%81%AE%E6%BC%94%E7%BF%92%EF%BC%88BikeSharingDemand%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#回帰分析の演習



## 使用データ

以下に示すKaggleのデータは、
Washington, D.C. におけるあるバイクシェアリングサービスでの実績データである。


Kaggle: **Bike Sharing Demand**  
https://www.kaggle.com/competitions/bike-sharing-demand

このデータの取得には、**コンペへの参加**
が必要である。
上のリンク先のページ内の
`Late Submission` をクリックしてコンペに参加する。
ただし、すでにコンペの締め切りを過ぎているので、
賞金がもらえるわけではない。

データはZIPファイルをダウンロードして、
 `train.csv` と `test.csv` を取り出す。

> **重要**：`count = casual + registered` であるため、`count`を予測するモデルに `casual` と `registered` を入れてはいけない。

##データセットの概要

**Bike Sharing Demand** は、自転車シェアリングサービスの**1時間ごとのレンタル需要**を予測するためのデータである。

Kaggleの公式説明では、約2年間にわたる時間単位のレンタル実績が提供されており、日時、季節、休日・平日、天候、気温、湿度、風速などの情報から、その時間帯に借りられる自転車の総数 `count` を予測する。

各行は、

> **「ある日時の1時間について、そのときの暦・天候条件と実際の自転車利用数を記録したもの」**

である。本データは単なる一般的な回帰データではなく、

- 時刻による需要変動
- 曜日・休日による需要変動
- 季節変動
- 天候による需要変動
- 年をまたぐ需要水準の変化

などを含む**時系列性を持つ需要予測データ**である。

##予測対象

本演習の目的変数は `count` であり、**その時間帯にレンタルされた自転車の総数**を表す。

データには次の2種類の利用者数も含まれている。

- `casual`：非登録利用者によるレンタル数
- `registered`：登録利用者によるレンタル数

なお、

$$
\text{count} = \text{casual} + \text{registered}
$$

という関係がある。

したがって、`count` を予測するときに `casual` と `registered` を説明変数として使用すると、答えを構成する情報を入力してしまうことになる点に注意が必要である。


## 主な変数

| 変数 | 種類 | 内容 | 本演習での扱い |
|---|---|---|---|
| `datetime` | 日時 | 観測された年月日と時刻 | 年・月・時刻・曜日などを作る元データ |
| `season` | カテゴリ | 1=春、2=夏、3=秋、4=冬 | 説明変数 |
| `holiday` | 0/1 | 祝日かどうか | 説明変数 |
| `workingday` | 0/1 | 土日・祝日ではない日か | 説明変数 |
| `weather` | カテゴリ | 天候の状態を1～4で表現 | 説明変数 |
| `temp` | 数値 | 気温（℃） | 説明変数 |
| `atemp` | 数値 | 体感気温（℃） | 説明変数 |
| `humidity` | 数値 | 相対湿度 | 説明変数 |
| `windspeed` | 数値 | 風速 | 説明変数 |
| `casual` | 数値 | 非登録利用者のレンタル数 | **説明変数に使用してはならない** |
| `registered` | 数値 | 登録利用者のレンタル数 | **説明変数に使用してはならない** |
| `count` | 数値 | 総レンタル数 | **目的変数** |

---

### `weather` の意味

Kaggleでは天候を次の4段階で表している。

| 値 | おおよその意味 |
|---:|---|
| 1 | 晴れ、少ない雲、部分的な曇り |
| 2 | 霧、曇りなど |
| 3 | 小雪、小雨、雷雨を伴う弱い降水など |
| 4 | 強い雨、雪、霧を伴う悪天候など |

---

### 時間によって需要が変わることが重要

このデータでは、同じ気温・天候でも、

- 朝8時
- 昼12時
- 夕方18時
- 深夜0時

では利用目的が異なり、需要も大きく変わる可能性がある。
また、平日と休日、夏と冬、年度の違いでも需要構造が変わり得る。

---

### Kaggleでの `train.csv` / `test.csv` の分割

Kaggle公式コンペでは、単純なランダム分割ではない。

各月について、

- **1日～19日**：`train.csv`
- **20日～月末**：`test.csv`

という構成になっている。

そして `test.csv` の各時間帯について、総レンタル数 `count` を予測する。


---
---
#演習のテーマ
この演習の目的は、`count` の予測を行う**回帰モデル**を構築することである。

Kaggleと同様に、
`test.csv` に格納された
日時、季節、休日・平日、天候、気温、湿度、風速などの情報から、その時間帯に借りられる自転車の総数 `count` を予測するモデルを構築する。

学習データとしては、`train.csv` を用いて、データの前処理、モデル構築と精度評価を行い、その結果をレポートする。